## Warm-up: re-drill from memory (5 min, no scrolling up, no old notebooks)

**A.** In the blank cell below, load `../data/loans_messy.csv` and type the full **first-look inspection checklist** from memory (7 items). Run it.

**B.** One line each, time AND space, in proper notation (biggest term only):
- P3 Valid Parentheses (stack):
- P4 Best Time to Buy/Sell (one pass, track min):
- P6 First Unique Char (Counter over lowercase letters):

In [ ]:
import pandas as pd
# your code here


## Concept primer: window functions + CTEs

**A window function computes a value per row using other rows, without collapsing them.**
`GROUP BY` turns many rows into one. `OVER (...)` keeps every row and adds a column.

```sql
func(...) OVER (PARTITION BY <group> ORDER BY <sort>)
```
- `PARTITION BY` = "restart for each group" (like groupby). Leave it out = whole table is one window.
- `ORDER BY` inside `OVER` = order used for ranking / running totals / LAG. Separate from the final `ORDER BY`.

| Function | Use it when the question says | Example |
|---|---|---|
| `ROW_NUMBER()` | "the top 1 / latest / first per group" | 1,2,3,4 (ties broken arbitrarily) |
| `RANK()` | "top N, ties share a place" | 1,2,2,4 (gap after tie) |
| `DENSE_RANK()` | "Nth highest distinct value" | 1,2,2,3 (no gap) |
| `LAG(col)` / `LEAD(col)` | "previous / next row", "month-on-month change" | value from row before/after |
| `SUM(col) OVER (... ORDER BY ...)` | "running total", "cumulative" | adds up to the current row |

**Key trap:** window functions run in `SELECT`, which comes *after* `WHERE`.
So `WHERE ROW_NUMBER() OVER (...) = 1` is an error. Compute the rank in a **CTE**, filter in the outer query:

```sql
WITH ranked AS (                          -- CTE = a named temporary result
    SELECT region, customer_id, income,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY income DESC) AS rn
    FROM customers
)
SELECT * FROM ranked WHERE rn = 1;        -- richest customer per region
```

**Tiny example:** scores `[90, 80, 80, 70]` ordered DESC: ROW_NUMBER `1,2,3,4` · RANK `1,2,2,4` · DENSE_RANK `1,2,2,3`.

## Problem: window functions on the bank database

Write each query as a string, look at it with `run(...)`, then `check(...)`.
**Use exactly the columns and `ORDER BY` asked for.** Floats are compared at 2 dp.
Target: **25 minutes** for Q1-Q4. Note your start time.

In [ ]:
import sqlite3, hashlib
import pandas as pd
con = sqlite3.connect("../data/bank.db")

def run(sql):
    """Run a query and show the result as a table."""
    return pd.read_sql(sql, con)

_ANS = {"q1": "cc48dbdb21", "q2": "164d0b5a44", "q3": "d1ea78f66a", "q4": "1afad47f57"}
def check(key, sql):
    """Checks your query's rows (values AND order) against the hidden answer."""
    rows = con.execute(sql).fetchall()
    norm = [tuple(round(x, 2) if isinstance(x, float) else x for x in r) for r in rows]
    ok = hashlib.md5(repr(norm).encode()).hexdigest()[:10] == _ANS[key]
    print(f"{key}: {'CORRECT' if ok else 'WRONG'}  ({len(rows)} rows)")

## Schema map

```
customers (200 rows)                 one row = one customer
  customer_id  PK ─────────┬──────────────────────┐
  first_name, last_name    │                      │
  segment (NULLs), region  │                      │
  annual_income (NULLs)    │                      │
                           │ 1 : many             │ 1 : many
                           ▼                      ▼
accounts (372)                          loans (158)
  account_id   PK                         loan_id      PK      one row = one loan
  customer_id  FK → customers             customer_id  FK → customers
  account_type, balance, status           loan_type, principal, status, credit_score
       │ 1 : many
       ▼
transactions (9,867)                    one row = one transaction
  txn_id       PK
  account_id   FK → accounts
  txn_ts 'YYYY-MM-DD HH:MM:SS', amount (neg = debit), channel, is_fraud
```

10 customers have no accounts; ~15% of accounts have no transactions.
Month from a timestamp: `substr(txn_ts, 1, 7)` → `'2025-03'`. Columns of one table: `run("PRAGMA table_info(loans)")`.

**Q1. Top 1 per group.** For each `loan_type`, return the single **largest loan by `principal`**.
Columns: `loan_type, loan_id, customer_id, principal`. Order by `loan_type`.

In [ ]:
q1 = """

"""
run(q1)


In [ ]:
check("q1", q1)

**Q2. Top N per group with ties.** For each `region`, find the **top 3 customers by total balance** (sum of `balance` over all their accounts, rounded to 2 dp). Ties share a rank.
Columns: `region, customer_id, total_balance, rnk`. Order by `region`, `rnk`, `customer_id`.

In [ ]:
q2 = """

"""
run(q2)


In [ ]:
check("q2", q2)

**Q3. Month-on-month change.** Total **debit spend** per month (debits only, `amount < 0`, shown as a positive number, rounded to 2 dp; include all transactions, fraud too). Add the previous month's spend and the % change from the previous month (rounded to 1 dp, computed from the rounded spends). First month has NULLs.
Columns: `month, spend, prev_spend, pct_change`. Order by `month`.

In [ ]:
q3 = """

"""
run(q3)


In [ ]:
check("q3", q3)

**Q4. Running total.** For accounts **1001 and 1002**, list every transaction with a **running total of `amount` within each account**, in time order (break timestamp ties by `txn_id`). Round the running total to 2 dp.
Columns: `account_id, txn_id, txn_ts, amount, running_total`. Order by `account_id`, `txn_ts`, `txn_id`.

In [ ]:
q4 = """

"""
run(q4)


In [ ]:
check("q4", q4)

## Explain it

1. Why can't you put `ROW_NUMBER() ... = 1` straight in `WHERE`? (Use the order of execution.)
2. In Q1, if two loans of the same type tied on principal, what would `ROW_NUMBER` vs `RANK` return?
3. `GROUP BY` vs `PARTITION BY`: one sentence on the difference in output row count.
4. Q3: write the business sentence you'd give a manager about the biggest monthly swing, plus one caveat.

_your answers here_